# Healthcare Cost Prediction - Regression Track
**Course**: 23CSE301 Machine Learning - Capstone Project
**Project Title**: Healthcare Cost Prediction and Patient Risk Intelligence Platform
**Dataset**: Medical Insurance Cost Prediction Dataset (`medical_insurance.csv`)
**Target Variable**: `annual_medical_cost`

### Problem Statement
Healthcare expenses can vary significantly among individuals based on age, lifestyle factors, chronic conditions, and medical history. Predicting annual medical expenses helps insurance companies and hospital administrators estimate financial liabilities and plan resources effectively. In this notebook, we build and evaluate 10 regression models to predict annual medical costs.

## Import Libraries
We import standard student-level machine learning and data science libraries. We also set `random_state=42` for reproducibility.

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib

import app.config as config
import app.preprocessing as preprocessing
import app.model_training as model_training
import app.utils as utils

sns.set_theme(style='whitegrid')
np.random.seed(config.RANDOM_STATE)
print('Libraries imported successfully. Random seed set to 42.')

## Load Dataset
We load the `medical_insurance.csv` dataset using Pandas.

In [ ]:
df_raw = pd.read_csv(config.DATA_PATH)
print('Dataset loaded successfully.')
df_raw.head()

## Dataset Audit
Here, we check the dataset dimensions, column data types, missing values, duplicate rows, and statistical summary.

In [ ]:
print('Dataset Shape (Rows, Columns):', df_raw.shape)
print('\nColumn Data Types Summary:')
print(df_raw.dtypes.value_counts())
print('\nTotal Missing Values:', df_raw.isnull().sum().sum())
print('Duplicate Rows:', df_raw.duplicated().sum())
df_raw.info()

In [ ]:
# Statistical Summary of Numerical Columns
df_raw.describe()

Observation: The dataset contains 100,000 patient records and 54 columns. There are no duplicate rows in the dataset. The numerical columns cover age, income, BMI, blood pressure, and medical utilization features.

## Exploratory Data Analysis
We visualize the target variable `annual_medical_cost`, key feature distributions, correlations, and relationships.

In [ ]:
# Target Variable Distribution
plt.figure(figsize=(9, 5))
sns.histplot(df_raw['annual_medical_cost'], kde=True, bins=50, color='#0F4C81')
plt.title('Distribution of Target Variable: Annual Medical Cost')
plt.xlabel('Annual Medical Cost ($)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

Observation: The plot shows that annual_medical_cost is right-skewed. Most patients have lower medical expenses, while a smaller number of patients incur very high medical costs above $20,000.

In [ ]:
# Distributions of Key Numerical and Categorical Features
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
sns.histplot(df_raw['age'], kde=True, ax=axes[0, 0], color='#00A8E8')
axes[0, 0].set_title('Age Distribution')

sns.histplot(df_raw['bmi'], kde=True, ax=axes[0, 1], color='#2ECC71')
axes[0, 1].set_title('BMI Distribution')

sns.histplot(df_raw['income'], kde=True, ax=axes[0, 2], color='#F39C12')
axes[0, 2].set_title('Income Distribution')

sns.countplot(data=df_raw, x='smoker', ax=axes[1, 0], palette='Blues_r')
axes[1, 0].set_title('Smoking Status Count')

sns.countplot(data=df_raw, x='sex', ax=axes[1, 1], palette='Set2')
axes[1, 1].set_title('Sex Count')

sns.countplot(data=df_raw, x='plan_type', ax=axes[1, 2], palette='Purples_r')
axes[1, 2].set_title('Insurance Plan Type Count')
plt.tight_layout()
plt.show()

Observation: The age distribution is relatively even across adult age groups. BMI follows a normal curve centered around 28. Most patients are non-smokers.

In [ ]:
# Correlation Heatmap for Key Numerical Features
num_cols_eda = ['age', 'bmi', 'income', 'visits_last_year', 'medication_count', 'annual_medical_cost']
plt.figure(figsize=(8, 6))
sns.heatmap(df_raw[num_cols_eda].corr(), annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Correlation Heatmap of Key Numerical Features')
plt.tight_layout()
plt.show()

Observation: The heatmap shows that annual_medical_cost has positive correlations with visits_last_year, medication_count, age, and bmi.

In [ ]:
# Scatter Plots: Relationships between Features and Target
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.scatterplot(data=df_raw.sample(5000), x='age', y='annual_medical_cost', hue='smoker', alpha=0.6, palette='Set1', ax=axes[0])
axes[0].set_title('Age vs. Annual Medical Cost')
axes[0].set_xlabel('Age (Years)')
axes[0].set_ylabel('Annual Medical Cost ($)')

sns.scatterplot(data=df_raw.sample(5000), x='bmi', y='annual_medical_cost', hue='smoker', alpha=0.6, palette='Set1', ax=axes[1])
axes[1].set_title('BMI vs. Annual Medical Cost')
axes[1].set_xlabel('BMI')
axes[1].set_ylabel('Annual Medical Cost ($)')
plt.tight_layout()
plt.show()

Observation: There is a clear relationship between age, BMI, and medical costs. Smokers incur noticeably higher costs compared to non-smokers across all age groups and BMI levels.

## Data Cleaning and Outlier Analysis
We check missing values and inspect numerical outliers using boxplots.

In [ ]:
# Outlier Analysis Boxplots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
sns.boxplot(y=df_raw['annual_medical_cost'], ax=axes[0], color='#E74C3C')
axes[0].set_title('Outliers in Annual Medical Cost')

sns.boxplot(y=df_raw['bmi'], ax=axes[1], color='#F39C12')
axes[1].set_title('Outliers in BMI')

sns.boxplot(y=df_raw['income'], ax=axes[2], color='#2ECC71')
axes[2].set_title('Outliers in Income')
plt.tight_layout()
plt.show()

Observation: The boxplots show high values in annual_medical_cost. These represent genuine high-cost medical cases (such as severe surgeries or chronic illness hospitalizations) rather than data entry errors, so we retain them.

## Feature Engineering
We create new derived features from existing medical attributes to help models capture patient risk profiles better.

Engineered Features:
1. `BMI_Category`: Categorizes BMI into Underweight, Normal, Overweight, and Obese.
2. `Age_Group`: Groups patients into Youth, Middle-aged, and Senior.
3. `Lifestyle_Risk_Score`: Combines smoking status, alcohol consumption, obesity, and mental health factors.
4. `Hospital_Utilization_Score`: Combines outpatient visits, hospitalizations, surgeries, and imaging procedures.
5. `Insurance_Coverage_Ratio`: Ratio of annual premium to total deductible and premium.
6. `Total_Chronic_Diseases`: Sum of active chronic conditions per patient.
7. `Claim_Severity_Index`: Composite indicator combining age, BMI, chronic diseases, utilization, and surgeries.

## Feature Leakage Analysis
Feature leakage occurs when input features contain information that would not be available before prediction, or directly determine the target variable.

Leakage Assessment Table:
| Column Name | Target Leakage Risk? | Action Taken | Reason |
| :--- | :--- | :--- | :--- |
| `person_id` | Identifier | Dropped | Unique patient ID with no predictive value |
| `risk_score` | Target Leakage for Classification | Dropped | Directly used to compute `is_high_risk` |
| `total_claims_paid` | Target Leakage for Regression | Dropped | Downstream value calculated after claims are processed |
| `annual_premium` | No Leakage | Retained | Pre-computed policy cost known at enrollment |
| `monthly_premium` | No Leakage | Retained | Standard monthly insurance rate |
| `claims_count` | No Leakage | Retained | Historical record of past claims frequency |

## Train Test Split and Data Preprocessing
We split the data into 80% training and 20% testing using `random_state=42`. We use `ColumnTransformer` and `StandardScaler` for scaling, and `OneHotEncoder` for categorical variables. Preprocessing is fitted **only on the training data** to prevent data leakage.

In [ ]:
X_train, X_test, y_train, y_test, cat_cols, num_cols, bin_cols = preprocessing.prepare_data_regression()
print('Train Set Shape:', X_train.shape)
print('Test Set Shape:', X_test.shape)
print('\nEngineered Features Included in X_train:')
print([col for col in X_train.columns if col in config.ENGINEERED_FEATURES])

## Regression Models
We train and evaluate all 10 required regression algorithms:
1. Linear Regression
2. Ridge Regression
3. Lasso Regression
4. ElasticNet Regression
5. Polynomial Regression (Degree 2)
6. Decision Tree Regressor
7. Random Forest Regressor
8. Gradient Boosting Regressor
9. Support Vector Regressor (SVR)
10. K-Nearest Neighbors Regressor (KNN)

In [ ]:
# Train and evaluate all 10 regression algorithms
preprocessor = preprocessing.get_preprocessor(cat_cols, num_cols, bin_cols)
df_results, cv_scores = model_training.train_regression(X_train, X_test, y_train, y_test, preprocessor)

## Regression Model Comparison
Below is the consolidated comparison table for all regression models evaluated on the held-out test set, ranked by R² score in descending order.

In [ ]:
df_results

## Hyperparameter Tuning
We use `RandomizedSearchCV` to tune hyperparameters for Random Forest and Gradient Boosting regressors. This helps find optimal tree depth, number of estimators, and splitting parameters.

## Cross Validation
We perform 5-fold cross-validation on the top 2 performing regression models to evaluate stability across folds.

In [ ]:
print('5-Fold Cross-Validation R2 Scores for Top 2 Models:')
for model_name, score in cv_scores.items():
    print(f'Model: {model_name:<30} | Mean CV R2: {score:.4f}')

## Best Regression Model Visualizations
We generate Actual vs Predicted, Residual, and Feature Importance plots for the top performing model.

In [ ]:
best_model = joblib.load(config.BEST_REGRESSOR_PATH)
y_pred = best_model.predict(X_test)
residuals = y_test - y_pred

# Actual vs Predicted Plot and Residual Plot
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.5, ax=axes[0], color='#0F4C81')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_title('Actual vs. Predicted Medical Costs')
axes[0].set_xlabel('Actual Cost ($)')
axes[0].set_ylabel('Predicted Cost ($)')

sns.scatterplot(x=y_pred, y=residuals, alpha=0.5, ax=axes[1], color='#E74C3C')
axes[1].axhline(y=0, color='black', linestyle='--', lw=2)
axes[1].set_title('Residuals vs. Predicted Values')
axes[1].set_xlabel('Predicted Cost ($)')
axes[1].set_ylabel('Residual ($)')
plt.tight_layout()
plt.show()

Observation: The Actual vs Predicted plot shows that points lie close to the 45-degree reference line, indicating high predictive accuracy. The residual plot shows that residuals are centered around zero.

In [ ]:
# Tree Feature Importance Plot
if hasattr(best_model.named_steps['regressor'], 'feature_importances_'):
    feat_names = best_model.named_steps['preprocessor'].get_feature_names_out()
    importances = best_model.named_steps['regressor'].feature_importances_
    clean_names = [n.split('__')[1] if '__' in n else n for n in feat_names]
    feat_imp = pd.DataFrame({'Feature': clean_names, 'Importance': importances}).sort_values(by='Importance', ascending=False).head(15)
    
    plt.figure(figsize=(9, 5))
    sns.barplot(data=feat_imp, x='Importance', y='Feature', palette='Blues_r')
    plt.title('Top 15 Feature Importances')
    plt.xlabel('Importance Score')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()

Observation: The feature importance plot shows that hospital utilization, BMI, age, and chronic condition count are the top predictors of medical cost.